In [ ]:
import pandas as pd
import ast

df = pd.read_csv('file_name.txt', sep='\t')

# Очистка пробелов
df.columns = df.columns.str.strip()

# Приведение столбца Position к числовому типу
df['Position'] = pd.to_numeric(df['Position'], errors='coerce')
df = df.dropna(subset=['Position'])
df['Position'] = df['Position'].astype(int)

# Преобразование base_counts в кортежи
def parse_counts(s):
    try: return ast.literal_eval(s)
    except: return (0, 0, 0, 0)

df['counts_tuple'] = df['BaseCount(A,C,G,T)'].apply(parse_counts)

# Функция для расчета
def get_heteroplasmy(pos, mut_nuc):
    row = df[df['Position'] == pos]
    if row.empty:
        return None

    counts = row.iloc[0]['counts_tuple']
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    idx = mapping.get(mut_nuc)

    if idx is None: return None

    mut_count = counts[idx]
    total_cov = row.iloc[0]['Cov']

    return (mut_count / total_cov) * 100 if total_cov > 0 else 0

# Список мутаций
mutations = {
    11778: 'A', 3460: 'A', 14484: 'C', 1555: 'G', 3376: 'A',
    3635: 'A', 3697: 'A', 3700: 'A', 3733: 'A', 4171: 'A',
    10197: 'A', 10663: 'C', 13051: 'A', 13094: 'C', 14459: 'A',
    14482: 'A', 14482: 'G', 14495: 'G', 14502: 'C', 14568: 'T'
}

# Вывод с фильтрацией
print(f"{'Вариант':<15} | {'Гетероплазмия':<15}")
print("-" * 35)

for pos, nuc in mutations.items():
    level = get_heteroplasmy(pos, nuc)

    if level is not None and level >= 1.0:
        row = df[df['Position'] == pos]
        ref_nuc = row['RefNuc'].values[0] if not row.empty else "?"

        print(f"m.{pos}{ref_nuc}>{nuc:<6} : {level:.2f}%")

Вариант         | Гетероплазмия  
-----------------------------------
m.3460G>A      : 1.33%
m.3635G>A      : 1.10%
